# logit-thermo A2-7：滴定の分析（CPU）

- **必要な secret：なし**。**ランタイム：CPU（標準）**。見込みは 15〜30 分。**データ集めの 3 本が全部終わってから**回してください。
- `A2_4/` の「なし」と n ＝ 4（block）、`A2_7/` の n ＝ 8・6・3・2 を読み、判定を計算して `A2_7/a2_7_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：A2-2・A2-4・A2-5 の分析のコードは Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_7_analyze.py`（`898b48ff715f2fe3…`）は埋め込み（データを開く前に固定）。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo A2-7 滴定 分析（判定と要約、CPU）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-7 滴定 分析（判定と要約、CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-7 analyze', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：分析のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRCA22': 'fe1f96db1f7e84031e85a70bcc8dcbb95c9fc06b3367657ff4074a9f9d1343f7', 'SRCA24': 'e70bad0db9b7780c2059aae52ad06debe87757bc582791deefb58c5f435f84a6', 'SRC5A': 'f4ef35b9f594a205ac1145c1a326e9de8a25280171fe2b93083cc1676e58d3ae'})
SRCA22, SRCA24, SRC5A = _s['SRCA22'], _s['SRCA24'], _s['SRC5A']
SRC7A = '"""A2-7（ゲート docs/A2_7_GATE.md）の分析：滴定。データを開く前に固定する。CPU。\nscripts/a2_2_analyze.py（pav・crossing_mono・feedback_stats・per_doc_feedback・NBOOT）、scripts/a2_4_analyze.py（stats_fixed・paired・\nverdict_inf・copy_frac）、scripts/a2_5_analyze.py（spearman）を実行したのと同じ名前空間で実行する。\n\n濃さの並び（薄い → 濃い）：なし・n8・n6・n4・n3・n2。なし と n4 は A2-4（A2_4/{モデル}@none.npz・@block.npz）の、A7_TEMPS の 8 つの温度だけを使う。\nε はモデルごとに 1 つ：「なし」の 8 つの温度のデータで feedback_stats が出す ε。\n"""\nimport math\nimport numpy as np\n\nA7_MODELS = [\'pythia-1b\', \'olmo2-1b\', \'qwen2.5-1.5b\']\nA7_DOSES = [\'none\', \'n8\', \'n6\', \'n4\', \'n3\', \'n2\']\nA7_TEMPS_A = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05]\nA7_BOOT, A7_BOOT_SEED = 1000, 20261107\nA7_FLOOR_T = 0.6          # 「大きく下がる（格子の外）」のときに、並びの計算で使う T_0（格子の下の端）\n\n\ndef load_fds(d4, d7, m):\n    Zn = np.load(f\'{d4}/{m}@none.npz\')\n    t4 = [float(t) for t in Zn[\'temps\']]\n    kk = [t4.index(t) for t in A7_TEMPS_A]\n    tf = Zn[\'tf\'][0][:, kk]                                        # (nD, 8, L)\n    use = [int(u) for u in Zn[\'use\']]\n    fds, extra = {}, {}\n    for dose in A7_DOSES:\n        if dose in (\'none\', \'n4\'):\n            Z = Zn if dose == \'none\' else np.load(f\'{d4}/{m}@block.npz\')\n            g = Z[\'g_ell\'][kk]\n            toks = Z[\'toks\'][kk]\n        else:\n            Z = np.load(f\'{d7}/{m}@{dose}.npz\')\n            assert [float(t) for t in Z[\'temps\']] == A7_TEMPS_A, \'温度の並びが違います\'\n            g = Z[\'g_ell\']; toks = Z[\'toks\']\n        assert [int(u) for u in Z[\'use\']] == use, (\'使った記事が違います\', m, dose)\n        fds[dose] = per_doc_feedback(tf, g)\n        extra[dose] = dict(toks=toks, prompts=Z[\'prompts\'])\n    return fds, extra\n\n\ndef dose_T0(fd, eps):\n    s = stats_fixed(fd, A7_TEMPS_A, eps)\n    T0 = s[\'T0\']\n    rank_T0 = T0 if np.isfinite(T0) else (A7_FLOOR_T if s[\'cls\'] == \'below\' else float(\'nan\'))\n    return s, rank_T0\n\n\ndef analyze7(d4, d7):\n    res = {\'models\': {}}\n    for m in A7_MODELS:\n        fds, extra = load_fds(d4, d7, m)\n        eps = feedback_stats(fds[\'none\'], A7_TEMPS_A, np.random.default_rng(0))[\'eps\']\n        row = {\'eps\': eps, \'doses\': {}}\n        ys = []\n        for dose in A7_DOSES:\n            s, rT = dose_T0(fds[dose], eps)\n            k07 = A7_TEMPS_A.index(0.7)\n            toks, prm = extra[dose][\'toks\'][k07], extra[dose][\'prompts\']\n            c4 = float(np.mean([copy_frac(prm[j], toks[s_, j], 4) for s_ in range(toks.shape[0]) for j in range(toks.shape[1])]))\n            c3 = float(np.mean([copy_frac(prm[j], toks[s_, j], 3) for s_ in range(toks.shape[0]) for j in range(toks.shape[1])]))\n            d = dict(T0=s[\'T0\'], cls=s[\'cls\'], T0_rank=rT, copy4_07=c4, copy3_07=c3)\n            if dose != \'none\':\n                p = paired(fds[dose], fds[\'none\'], A7_TEMPS_A, eps, seed=1)\n                d.update(d_vs_none=p, word=verdict_inf(p, s[\'cls\'], True))\n            row[\'doses\'][dose] = d\n            ys.append(rT)\n        # TQ2：濃くするほど T_0 が下がるか（6 点の並び）\n        xs = list(range(len(A7_DOSES)))\n        r = spearman(xs, ys) if np.all(np.isfinite(ys)) else float(\'nan\')\n        rng = np.random.default_rng(A7_BOOT_SEED)\n        nD = fds[\'none\'].shape[0]\n        neg, okn = 0, 0\n        for _ in range(A7_BOOT):\n            idx = rng.integers(0, nD, nD)\n            yb = [dose_T0(fds[dz][idx], eps)[1] for dz in A7_DOSES]\n            if np.all(np.isfinite(yb)):\n                okn += 1\n                neg += int(spearman(xs, yb) < 0)\n        ok_frac = okn / A7_BOOT\n        word = (\'濃いほど下がる\' if (np.isfinite(r) and r <= -0.9 + 1e-9 and ok_frac >= 0.9 and neg / A7_BOOT >= 0.95)\n                else \'はっきりしない\')\n        row.update(rho=r, boot_ok=ok_frac, frac_neg=neg / A7_BOOT, TQ2_word=word)\n        res[\'models\'][m] = row\n    res[\'TQ1_olmo\'] = res[\'models\'][\'olmo2-1b\'][\'doses\'][\'n2\'][\'word\']\n    res[\'TQ1_qwen\'] = res[\'models\'][\'qwen2.5-1.5b\'][\'doses\'][\'n2\'][\'word\']\n    res[\'TQ1_pythia_record\'] = res[\'models\'][\'pythia-1b\'][\'doses\'][\'n2\'][\'word\']\n    res[\'TQ2\'] = sum(res[\'models\'][m][\'TQ2_word\'] == \'濃いほど下がる\' for m in A7_MODELS)\n    return res\n\n\ndef self_test_a27_analyze():\n    # 作り物：T_0 が濃さとともに下がる記事ごとの f（8 温度）。濃さごとに交点を 0.86 → 0.70 へ\n    rng = np.random.default_rng(0)\n    t = np.array(A7_TEMPS_A)\n    def fd_of(T0, n=40, noise=0.05):\n        F = np.where(t < T0, (t - T0) * 12, (t - T0) * 5)\n        return F[None, :] + rng.normal(0, noise, (n, len(t)))\n    T0s = [0.86, 0.84, 0.81, 0.78, 0.74, 0.70]\n    fds = {d: fd_of(T) for d, T in zip(A7_DOSES, T0s)}\n    eps = feedback_stats(fds[\'none\'], A7_TEMPS_A, np.random.default_rng(0))[\'eps\']\n    ys = [dose_T0(fds[d], eps)[1] for d in A7_DOSES]\n    assert spearman(list(range(6)), ys) <= -0.9, ys\n    p = paired(fds[\'n2\'], fds[\'none\'], A7_TEMPS_A, eps, seed=1)\n    assert verdict_inf(p, \'cross\', True) == \'大きく下がる\', p\n    # 格子の外（全部の温度で F > ε）は T_0 ＝ 0.6 として並べる\n    s, rT = dose_T0(fd_of(0.4), eps)\n    assert s[\'cls\'] == \'below\' and rT == A7_FLOOR_T\n    return \'OK\'\n'
assert hashlib.sha256(SRC7A.encode()).hexdigest() == '898b48ff715f2fe3a8c1431ff86fc99a19594500b2dc314f132641f858b3f4f5'
exec(SRCA22)
exec(SRCA24)
exec(SRC5A)
exec(SRC7A)
print('自己テスト：', self_test_a27_analyze())


In [ ]:
# セル3：分析して保存（結果は表示しない）
import json
try:
    r = analyze7('/content/drive/MyDrive/logit-thermo/A2_4', '/content/drive/MyDrive/logit-thermo/A2_7')
    r['a2_7_analyze_sha256'] = '898b48ff715f2fe3a8c1431ff86fc99a19594500b2dc314f132641f858b3f4f5'
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo/A2_7' + '/a2_7_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-7 の分析で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：判定と要約を A2_7/a2_7_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
